# Remap region ids

How to take coordinates or Allen ids from your own data and express them in a
different mapping. For what the mappings *are* and why Swanson behaves differently,
see [Brain region mappings](../explanation/brain-region-mappings.qmd).


In [1]:
# import brain atlas and brain regions objects
from iblatlas.atlas import AllenAtlas
from iblatlas.regions import BrainRegions
ba = AllenAtlas()
br = BrainRegions() # br is also an attribute of ba so could to br = ba.regions

## How to map your data

### Mapping from mlapdv coordinates
The recommended and most versatile way to find the locations of clusters under different mappings is to use the mlapdv coordinates of the clusters. Given a probe insertion id, the clusters object can be loaded in using the following code

In [2]:
from brainbox.io.one import SpikeSortingLoader
from one.api import ONE
one = ONE()

pid = 'da8dfec1-d265-44e8-84ce-6ae9c109b8bd'

sl = SpikeSortingLoader(pid=pid, one=one, atlas=ba)
spikes, clusters, channels = sl.load_spike_sorting()
clusters = sl.merge_clusters(spikes, clusters, channels)

You will find that the cluster object returned already contains an atlas_id attribute. These are the atlas ids that are obtained using the default mapping - **non lateralised Allen**. For this mapping regardless of whether the clusters lie in the right or left hemisphere the clusters are assigned positive atlas ids

In [3]:
print(clusters['atlas_id'][0:10])

[596 596 596 596 596 596 596 596 596 596]


We can obtain the mlapdv coordinates of the clusters and explore in which hemisphere the clusters lie, clusters with negative x lie on the left hemisphere.

In [4]:
import numpy as np
mlapdv = np.c_[clusters['x'], clusters['y'], clusters['z']] # x = ml, y = ap, z = dv
clus_LH = np.sum(mlapdv[:, 0] < 0)
clus_RH = np.sum(mlapdv[:, 0] > 0)
print(f'Total clusters = {len(mlapdv)}, LH clusters = {clus_LH}, RH clusters = {clus_RH}')

Total clusters = 1143, LH clusters = 1143, RH clusters = 0


To get a better understanding of the difference between using lateralised and non-lateralised mappings, let's also make a manipulated version of the mlapdv positions where the first 5 clusters have been moved into the right hemisphere and call this `mlapdv_rh`

In [5]:
mlapdv_rh = np.copy(mlapdv)
mlapdv_rh[0:5, 0] = -1 * mlapdv_rh[0:5, 0]
clus_LH = np.sum(mlapdv_rh[:, 0] < 0)
clus_RH = np.sum(mlapdv_rh[:, 0] > 0)
print(f'Total clusters = {len(mlapdv_rh)}, LH clusters = {clus_LH}, RH clusters = {clus_RH}')

Total clusters = 1143, LH clusters = 1138, RH clusters = 5


To find the locations of the clusters in the brain from the mlapdv position we can use the [get_labels](https://int-brain-lab.github.io/iblenv/_autosummary/ibllib.atlas.atlas.html#ibllib.atlas.atlas.BrainAtlas.get_labels) method in the AllenAtlas object. First let's explore the output of using a non lateralised mapping.

In [6]:
atlas_id_Allen = ba.get_labels(mlapdv, mapping='Allen')
atlas_id_Allen_rh = ba.get_labels(mlapdv_rh, mapping='Allen')
print(f'Non-lateralised Allen mapping ids using mlapdv: {atlas_id_Allen[0:10]}')
print(f'Non-lateralised Allen mapping ids using mlapdv_rh: {atlas_id_Allen_rh[0:10]}')

Non-lateralised Allen mapping ids using mlapdv: [596 596 596 596 596 596 596 596 596 596]
Non-lateralised Allen mapping ids using mlapdv_rh: [596 596 596 596 596 596 596 596 596 596]


Notice that regardless of whether the clusters lie in the left or right hemisphere the sign of the atlas id is the same. The result of this mapping is also equivalent the default output from `clusters['atlas_id']`

In [7]:
np.array_equal(clusters['atlas_id'], atlas_id_Allen)

True

Now if we use the lateralised mapping, we notice that the clusters in the left hemisphere have been assigned negative atlas ids whereas those in the right hemisphere have positive atlas ids

In [8]:
atlas_id_Allen = ba.get_labels(mlapdv, mapping='Allen-lr')
atlas_id_Allen_rh = ba.get_labels(mlapdv_rh, mapping='Allen-lr')
print(f'Lateralised Allen mapping ids using mlapdv: {atlas_id_Allen[0:10]}')
print(f'Lateralised Allen mapping ids using mlapdv_rh: {atlas_id_Allen_rh[0:10]}')

Lateralised Allen mapping ids using mlapdv: [-596 -596 -596 -596 -596 -596 -596 -596 -596 -596]
Lateralised Allen mapping ids using mlapdv_rh: [ 596  596  596  596  596 -596 -596 -596 -596 -596]


By changing the mapping argument that we pass in, we can also easily obtain the atlas ids for the Beryl and Cosmos mappings

In [9]:
atlas_id_Beryl = ba.get_labels(mlapdv_rh, mapping='Beryl-lr')
atlas_id_Cosmos = ba.get_labels(mlapdv_rh, mapping='Cosmos')
print(f'Lateralised Beryl mapping ids using mlapdv_rh: {atlas_id_Beryl[0:10]}')
print(f'Non-lateralised Cosmos mapping ids using mlapdv_rh: {atlas_id_Cosmos[0:10]}')

Lateralised Beryl mapping ids using mlapdv_rh: [ 596  596  596  596  596 -596 -596 -596 -596 -596]
Non-lateralised Cosmos mapping ids using mlapdv_rh: [623 623 623 623 623 623 623 623 623 623]


### Mapping from atlas ids
Methods are available that allow you to translate atlas ids from one mapping to another. 

In [10]:
# map atlas ids from lateralised Allen to lateralised Beryl
atlas_id_Allen = ba.get_labels(mlapdv_rh, mapping='Allen-lr') # lateralised Allen

remap_beryl = br.id2id(atlas_id_Allen, mapping='Beryl-lr')
print(f'Lateralised Beryl mapping ids using remap: {remap_beryl[0:10]}')

# map atlas ids from lateralised Allen to non-lateralised Cosmos
remap_cosmos = br.id2id(atlas_id_Allen_rh, mapping='Cosmos')
print(f'Non-lateralised Cosmos mapping ids using remap: {remap_cosmos[0:10]}')

Lateralised Beryl mapping ids using remap: [ 596  596  596  596  596 -596 -596 -596 -596 -596]
Non-lateralised Cosmos mapping ids using remap: [623 623 623 623 623 623 623 623 623 623]


When remapping with atlas ids it is not possible to map from 

1. A non-lateralised to a lateralised mapping. 
2. From a higher mapping to a lower one (e.g cannot map from Beryl to Allen, or Cosmos to Allen)
3. From Beryl to Cosmos

This is why it is recommened to use mlapdv coordinates for remappings as it allows complete flexibility

### Converting to acronyms
Methods are available to convert between atlas ids and acronyms. Note that when converting to acronyms, even if the atlas ids are lateralised the returned acronyms are not

In [11]:
atlas_id_Allen = ba.get_labels(mlapdv_rh, mapping='Allen-lr') # lateralised Allen
acronym_Allen = br.id2acronym(atlas_id_Allen, mapping='Allen-lr')
print(f'Acronyms of lateralised Allen ids: {acronym_Allen[0:10]}')

atlas_id_Allen = ba.get_labels(mlapdv_rh, mapping='Allen-lr') # Non-ateralised Allen
acronym_Allen = br.id2acronym(atlas_id_Allen)
print(f'Acronyms of non-lateralised Allen: {acronym_Allen[0:10]}')

Acronyms of lateralised Allen ids: ['NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB']
Acronyms of non-lateralised Allen: ['NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB' 'NDB']


It is also possible to simultaneously remap the acronyms with these methods

In [12]:
acronym_Cosmos = br.id2acronym(atlas_id_Allen, mapping='Cosmos-lr')
print(acronym_Cosmos[0:10])

['CNU' 'CNU' 'CNU' 'CNU' 'CNU' 'CNU' 'CNU' 'CNU' 'CNU' 'CNU']
